In [1]:
import pandas as pd
import json
import re

# Opening JSON file, with specific file name
fname = 'dev-SQuAD-v2.0'
f = open(fname + '.json')
#load in data as python dictionary using the json library
data = json.load(f)["data"]

In [2]:
#this is the dataframe that will be converted to pandas and then to csv
plaus = []
#loop through all different data titles/topics associated!
for i in range(0, len(data)):
    #get list of all paragraphs for later use
    paragraphs = data[i]['paragraphs']
    #loop through each paragraph
    for j in range(0, len(paragraphs)):
        #get all of the questions associated per paragraph
        questions = paragraphs[j]['qas']
        #get the context paragraph value
        context = paragraphs[j]['context']
        #loop through all questions in each topic paragraph
        for k in range(0, len(questions)):
            if(questions[k]['is_impossible'] == True):
                try:
                    plaus.append(questions[k]["plausible_answers"][0]['text'])
                except:
                    plaus.append("")
            

In [3]:
len(plaus)

5945

In [4]:
df = pd.read_csv("dev_joint_ds_all.csv")

In [5]:
df

,Unnamed: 0,qid,context,question,labels,answer span
0,0,56ddde6b9a695914005b9628,The Normans (Norman: Nourmands; French: Norman...,In what country is Normandy located?,Ans,"('France', 159)"
1,1,56ddde6b9a695914005b9629,The Normans (Norman: Nourmands; French: Norman...,When were the Normans in Normandy?,Ans,"('10th and 11th centuries', 94)"
2,2,56ddde6b9a695914005b962a,The Normans (Norman: Nourmands; French: Norman...,From which countries did the Norse originate?,Ans,"('Denmark, Iceland and Norway', 256)"
3,3,56ddde6b9a695914005b962b,The Normans (Norman: Nourmands; French: Norman...,Who was the Norse leader?,Ans,"('Rollo', 308)"
4,4,56ddde6b9a695914005b962c,The Normans (Norman: Nourmands; French: Norman...,What century did the Normans first gain their ...,Ans,"('10th century', 671)"
...,...,...,...,...,...,...
11868,11868,5737aafd1c456719005744ff,"The pound-force has a metric counterpart, less...",What is the seldom used force unit equal to on...,Ans,"('sthène', 665)"
11869,11869,5ad28ad0d7d075001a4299cc,"The pound-force has a metric counterpart, less...",What does not have a metric counterpart?,N,"('', '')"
11870,11870,5ad28ad0d7d075001a4299cd,"The pound-force has a metric counterpart, less...",What is the force exerted by standard gravity ...,E,"('', '')"
11871,11871,5ad28ad0d7d075001a4299ce,"The pound-force has a metric counterpart, less...",What force leads to a commonly used unit of mass?,E,"('', '')"


In [6]:
df = df[df.labels != "Ans"]

In [7]:
df

,Unnamed: 0,qid,context,question,labels,answer span
5,5,5ad39d53604f3c001a3fe8d1,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 1000's ...,I,"('', '')"
6,6,5ad39d53604f3c001a3fe8d2,The Normans (Norman: Nourmands; French: Norman...,What is France a region of?,I,"('', '')"
7,7,5ad39d53604f3c001a3fe8d3,The Normans (Norman: Nourmands; French: Norman...,Who did King Charles III swear fealty to?,I,"('', '')"
8,8,5ad39d53604f3c001a3fe8d4,The Normans (Norman: Nourmands; French: Norman...,When did the Frankish identity emerge?,E,"('', '')"
12,12,5ad3a266604f3c001a3fea27,"The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,X,"('', '')"
...,...,...,...,...,...,...
11863,11863,5ad28a57d7d075001a4299b3,The connection between macroscopic nonconserva...,What does not change macroscopic closed systems?,N,"('', '')"
11869,11869,5ad28ad0d7d075001a4299cc,"The pound-force has a metric counterpart, less...",What does not have a metric counterpart?,N,"('', '')"
11870,11870,5ad28ad0d7d075001a4299cd,"The pound-force has a metric counterpart, less...",What is the force exerted by standard gravity ...,E,"('', '')"
11871,11871,5ad28ad0d7d075001a4299ce,"The pound-force has a metric counterpart, less...",What force leads to a commonly used unit of mass?,E,"('', '')"


In [8]:
df.rename(columns={'labels': 'label', 'question':'original_question'}, inplace=True)
df["original_answer_span"] = plaus
def find_sentence(context, substring):
    substring = str(substring)
    # Split context into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', context)
    
    # Search for the substring in each sentence
    substring = str(substring)
    for sentence in sentences:
        if substring in sentence:
            return sentence
    return substring
def process(df):
    for index, row in df.iterrows():
        context = row['context']
        original_answer_span = row['original_answer_span']
        
        # Find the relevant sentence
        salience_sentence = find_sentence(context, original_answer_span)
        
        # Add the salience sentence to the DataFrame
        df.at[index, 'answer_sentence'] = salience_sentence
    return df
df = process(df)
df.to_csv("dev_sal.csv")

C:\Users\a1\AppData\Local\Temp\ipykernel_42716\24303507.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'labels': 'label', 'question':'original_question'}, inplace=True)
C:\Users\a1\AppData\Local\Temp\ipykernel_42716\24303507.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["original_answer_span"] = plaus
C:\Users\a1\AppData\Local\Temp\ipykernel_42716\24303507.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [9]:
df

,Unnamed: 0,qid,context,original_question,label,answer span,original_answer_span,answer_sentence
5,5,5ad39d53604f3c001a3fe8d1,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 1000's ...,I,"('', '')",Normans,The Normans (Norman: Nourmands; French: Norman...
6,6,5ad39d53604f3c001a3fe8d2,The Normans (Norman: Nourmands; French: Norman...,What is France a region of?,I,"('', '')",Normandy,The Normans (Norman: Nourmands; French: Norman...
7,7,5ad39d53604f3c001a3fe8d3,The Normans (Norman: Nourmands; French: Norman...,Who did King Charles III swear fealty to?,I,"('', '')",Rollo,"They were descended from Norse (""Norman"" comes..."
8,8,5ad39d53604f3c001a3fe8d4,The Normans (Norman: Nourmands; French: Norman...,When did the Frankish identity emerge?,E,"('', '')",10th century,The distinct cultural and ethnic identity of t...
12,12,5ad3a266604f3c001a3fea27,"The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,X,"('', '')","political, cultural and military","The Norman dynasty had a major political, cult..."
...,...,...,...,...,...,...,...,...
11863,11863,5ad28a57d7d075001a4299b3,The connection between macroscopic nonconserva...,What does not change macroscopic closed systems?,N,"('', '')",nonconservative forces,The connection between macroscopic nonconserva...
11869,11869,5ad28ad0d7d075001a4299cc,"The pound-force has a metric counterpart, less...",What does not have a metric counterpart?,N,"('', '')",pound-force,"The pound-force has a metric counterpart, less..."
11870,11870,5ad28ad0d7d075001a4299cd,"The pound-force has a metric counterpart, less...",What is the force exerted by standard gravity ...,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less..."
11871,11871,5ad28ad0d7d075001a4299ce,"The pound-force has a metric counterpart, less...",What force leads to a commonly used unit of mass?,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less..."
